# CONDITIONAL: управляем лучом, сохраняя один initial noise

**Target ray ID подаётся в каждый train / teacher / student / sampling вызов модели.**
Это отдельный conditional-пример на 16 вращающихся лучах. Unconditional-результаты
тетрадок 10–12 и их официальные конфигурации не меняются. Label приходит из dataset,
а не вычисляется после генерации. Координаты не вращаются, не обрезаются и не проецируются
на нужный луч после sampling.

Модель — обычный Diffusers `PointDenoiser`, `conditioner="embedding"`, hidden128 × 3.
Все FM- и distillation-loops открыты ниже. Нормированное время `t∈[0,1]` преобразуем
в `t*1000` **при каждом model call**: это соглашение PointDenoiser. `t=1` — шум,
`t=0` — данные. Веса, wheel и outputs не встроены в notebook.

Default загружает отдельно опубликованный conditional teacher **20 000 updates**.
`TRAIN_TEACHER=True` включает видимый FM-loop с нуля. Затем четыре student-модели
M0/M1/E0/E1 обучаются по 200 updates: 32→16, MSE / tube / Sinkhorn / их сочетание.
Одна condition назначается **всему OT support**; иначе matching мог бы соединять разные
запрошенные классы. С вероятностью .1 весь student-batch обучается как настоящий null
на смеси лучей. Общие RNG streams сохраняют paired-сравнение всех четырёх методов.

В конце **один и тот же шум повторяется для всех 16 запросов** и null. CFG по умолчанию
выключен: `guidance_scale=1` означает прямой conditional forward. `conditions=None` —
настоящий null forward, не случайный класс. Качество routing проверяется против
**запрошенного** ID; хорошая геометрия ближайшего произвольного луча этого не заменяет.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version

from packaging.version import Version

GIT_BRANCH = "research/wasserstein-tube-replication"
MIN_VERSION = "0.4.3"
GIT_SOURCE = ("diffusion-lab[replication,interactive] @ "
              "git+https://github.com/Vovanm88/ddpm_vibe_research.git@" + GIT_BRANCH)
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])
if Version(version("diffusion-lab")) < Version(MIN_VERSION):
    raise RuntimeError("Install diffusion-lab >=0.4.3 from this branch and restart the kernel")


In [ ]:
import copy
import hashlib
import json
import math
import tempfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, clear_output, display
from tqdm.auto import tqdm

from diffusion_lab import PointDenoiser
from diffusion_lab.conditional_metrics import conditional_routing_metrics
from diffusion_lab.conditional_visualization import (
    condition_routing_widget,
    plot_condition_routing,
    save_condition_routing_gif,
)
from diffusion_lab.environments import make_environment
from diffusion_lab.visualization import save_figure
from recheck.calibration import calibrate_interval_features
from recheck.fm import learning_rate
from recheck.losses import production_sinkhorn, sliced_wasserstein_loss
from recheck.runtime import (
    RNGStreams,
    atomic_json,
    choose_device,
    model_hash,
    save_checkpoint,
    tensor_hash,
)


In [ ]:
DEVICE = "cpu"
THREADS = 4
DATA_SEED = 47
VALIDATION_SEED = 10047
N_CLASSES = 16
N_TRAIN = 65536
N_VALIDATION = 8192
HIDDEN_DIM = 128
NUM_LAYERS = 3
CONDITIONER = "embedding"           # "one_hot" requires training a new teacher
TRAIN_TIMESTEPS = 1000

TRAIN_TEACHER = False
TEACHER_SEED = 47
TEACHER_UPDATES = 20000
TEACHER_BATCH_SIZE = 512
TEACHER_LR = 1e-3
TEACHER_WARMUP = 1000
TEACHER_EMA = .999
TEACHER_CONDITION_DROPOUT = .1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

SEED = 101
UPDATES = 200
BATCH_SIZE = 128                    # one condition for the whole OT support
LEARNING_RATE = 1e-4
WARMUP = 20
EMA_DECAY = .99
STUDENT_NULL_PROBABILITY = .1       # whole support is null on these updates
TEACHER_STEPS = 32
STUDENT_STEPS = 16
TUBE_RHO = .1
DISTRIBUTION_FAMILY = "sinkhorn"     # or "swd"
REPRESENTATION = "endpoint"         # or "displacement" / "joint"
LAMBDA_W = 1.0
GAMMA = 1.0
EPSILON_ALPHA = .05
CALIBRATION_SAMPLES = 256
CALIBRATION_SEED = 471300
SWD_PROJECTIONS = 128
CHECKPOINT_EVERY = 50
RESUME = False

SAMPLES_PER_TARGET = 128
EVAL_SEED = 471301
SAVE_GIFS = True
SHOW_WIDGETS = True
GIF_NOISE_INDEX = 0
PROGRESS = True
OUTPUT_DIR = Path("replication/artifacts/conditional_notebook13")
PUBLISHED_TEACHER_PATH = Path("replication/demo/rays16_conditional_v1/teacher/ema_model")


In [ ]:
def expected_teacher_profile():
    return {"data": {"name": "rotating_rays", "n_classes": N_CLASSES,
                      "n_train": N_TRAIN, "seed": DATA_SEED},
            "model": {"sample_shape": [2], "hidden_dim": HIDDEN_DIM,
                       "num_layers": NUM_LAYERS, "backbone": "mlp",
                       "conditioner": CONDITIONER, "num_classes": N_CLASSES},
            "teacher": {"seed": TEACHER_SEED, "updates": TEACHER_UPDATES,
                         "batch_size": TEACHER_BATCH_SIZE, "lr": TEACHER_LR,
                         "weight_decay": WEIGHT_DECAY, "warmup": TEACHER_WARMUP,
                         "ema_decay": TEACHER_EMA, "grad_clip": GRAD_CLIP,
                         "condition_dropout": TEACHER_CONDITION_DROPOUT}}


def load_conditional_parent(candidates, expected, device):
    rejected = []
    for candidate in candidates:
        candidate = Path(candidate)
        try:
            manifest = json.loads((candidate.parent / "manifest.json").read_text())
            native = json.loads((candidate / "config.json").read_text())
            if manifest.get("status") != "completed" or manifest.get("updates_completed", 0) < 20000:
                raise ValueError("conditional teacher has not completed 20000 updates")
            for section in ("data", "model", "teacher"):
                if manifest["config"][section] != expected[section]:
                    raise ValueError(f"wrong conditional teacher {section} profile")
            if native.get("_class_name") != "PointDenoiser":
                raise ValueError("expected native PointDenoiser")
            for key, value in expected["model"].items():
                if native.get(key) != value:
                    raise ValueError(f"wrong native conditional setting {key}")
            model = PointDenoiser.from_pretrained(candidate, local_files_only=True,
                low_cpu_mem_usage=False).to(device).eval().requires_grad_(False)
            if model_hash(model) != manifest["teacher_sha256"]:
                raise ValueError("conditional teacher native hash mismatch")
            if not manifest.get("data_sha256"):
                raise ValueError("missing train-data provenance")
            return model, candidate, manifest
        except (OSError, ValueError, KeyError, TypeError) as error:
            rejected.append(f"{candidate}: {error}")
    raise FileNotFoundError("Publish/train the matching conditional C16 teacher first, or set "
                            "TRAIN_TEACHER=True. No unconditional teacher is substituted.\n"
                            + "\n".join(rejected))


In [ ]:
repo_dir = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "src/diffusion_lab").is_dir()), None)
if repo_dir is None:
    repo_dir = Path(tempfile.mkdtemp(prefix="conditional-rays-source-")) / "repository"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                           "https://github.com/Vovanm88/ddpm_vibe_research.git", str(repo_dir)])
torch.set_num_threads(THREADS)
device = choose_device(DEVICE)
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = make_environment("rotating_rays", n_samples=N_TRAIN, seed=DATA_SEED, n_classes=N_CLASSES)
validation = make_environment("rotating_rays", n_samples=N_VALIDATION,
                              seed=VALIDATION_SEED, n_classes=N_CLASSES)
assert train.conditions.dtype == torch.long and train.conditions.shape == (N_TRAIN,)
class_indices = {label: torch.where(train.conditions == label)[0] for label in range(N_CLASSES)}
assert all(len(indices) for indices in class_indices.values())
assert TEACHER_STEPS % STUDENT_STEPS == 0 and STUDENT_STEPS > 0
MICROSTEPS, H = TEACHER_STEPS // STUDENT_STEPS, 1 / STUDENT_STEPS
if not 0 <= STUDENT_NULL_PROBABILITY <= 1:
    raise ValueError("STUDENT_NULL_PROBABILITY must be in [0,1]")
if REPRESENTATION not in {"endpoint", "displacement", "joint"}:
    raise ValueError("unknown representation")
profile = expected_teacher_profile()
print("CONDITIONAL dataset:", train.samples.shape, "labels:", train.conditions.unique().tolist())
print("Model receives target labels; no geometry relabeling or projection is used.")


## Видимый conditional flow-matching teacher

`labels` — реальные dataset ID. Per-example dropout обучает learned null-condition.
При `TRAIN_TEACHER=False` цикл пропускается: загружается отдельный native 20k teacher
с проверкой embedding/16 classes, полного профиля и hash. Новый training использует
`OUTPUT_DIR/teacher`; default exploratory-студенты не переписывают опубликованного родителя.

In [ ]:
if TRAIN_TEACHER:
    teacher_dir = OUTPUT_DIR / "teacher"
    if teacher_dir.exists() and any(teacher_dir.iterdir()):
        raise FileExistsError("Choose a new OUTPUT_DIR for a new teacher run")
    teacher_dir.mkdir(parents=True, exist_ok=True)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(TEACHER_SEED)
        teacher_model = PointDenoiser(**profile["model"]).to(device)
    teacher_ema = copy.deepcopy(teacher_model).eval().requires_grad_(False)
    teacher_optimizer = torch.optim.AdamW(teacher_model.parameters(), lr=TEACHER_LR,
        betas=(.9, .999), weight_decay=WEIGHT_DECAY)
    teacher_streams = RNGStreams(TEACHER_SEED)
    teacher_history = []
    atomic_json(teacher_dir / "manifest.json", {"status": "running", "config": profile,
        "updates_completed": 0, "data_sha256": tensor_hash(train.samples)})
    for update in tqdm(range(TEACHER_UPDATES), desc="Conditional FM teacher", disable=not PROGRESS):
        indices = torch.randint(len(train), (TEACHER_BATCH_SIZE,), generator=teacher_streams["data"])
        clean = train.samples[indices].to(device)
        labels = train.conditions[indices].to(device)
        noise = torch.randn(clean.shape, generator=teacher_streams["noise"]).to(device)
        t = torch.rand(len(clean), generator=teacher_streams["interval"]).to(device)
        keep = (torch.rand(len(clean), generator=teacher_streams["tube"])
                >= TEACHER_CONDITION_DROPOUT).to(device)
        anchor = (1 - t[:, None]) * clean + t[:, None] * noise
        target_velocity = noise - clean
        prediction = teacher_model(anchor, t * TRAIN_TIMESTEPS,
                                   conditions=labels, condition_mask=keep).sample
        loss = (prediction - target_velocity).square().mean()
        if not torch.isfinite(loss):
            raise FloatingPointError("Conditional teacher loss is nonfinite")
        lr = learning_rate(update, TEACHER_UPDATES, TEACHER_WARMUP, TEACHER_LR)
        for group in teacher_optimizer.param_groups:
            group["lr"] = lr
        teacher_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(teacher_model.parameters(), GRAD_CLIP, error_if_nonfinite=True)
        teacher_optimizer.step()
        with torch.no_grad():
            for average, parameter in zip(teacher_ema.parameters(), teacher_model.parameters()):
                average.lerp_(parameter.detach(), 1 - TEACHER_EMA)
            for average, buffer in zip(teacher_ema.buffers(), teacher_model.buffers()):
                average.copy_(buffer)
        teacher_history.append({"update": update + 1, "loss": float(loss.detach()), "lr": lr})
    teacher = teacher_ema.eval().requires_grad_(False)
    teacher_path = teacher_dir / "ema_model"
    teacher.save_pretrained(teacher_path, safe_serialization=True)
    teacher_manifest = {"status": "completed", "config": profile,
        "updates_completed": TEACHER_UPDATES, "teacher_sha256": model_hash(teacher),
        "data_sha256": tensor_hash(train.samples)}
    atomic_json(teacher_dir / "manifest.json", teacher_manifest)
    atomic_json(teacher_dir / "history.json", teacher_history)
else:
    teacher, teacher_path, teacher_manifest = load_conditional_parent([
        repo_dir / "replication/artifacts/rays16_conditional_v1/teacher/ema_model",
        repo_dir / PUBLISHED_TEACHER_PATH,
    ], profile, device)
    if teacher_manifest["data_sha256"] != tensor_hash(train.samples):
        raise ValueError("Conditional teacher and current labelled train data differ")
teacher_hash = model_hash(teacher)
assert all(p.grad is None and not p.requires_grad for p in teacher.parameters())
print("Native CONDITIONAL teacher:", teacher_path, "hash:", teacher_hash)


## Переход, CFG и sampling: condition остаётся частью каждого вызова

Tube меняет координаты anchor; **тот же label/null** передаётся каждому teacher microstep
из нового входа и student. CFG используется только при sampling. Выключенный CFG — одна
conditional ветка; включённый — `v_null + scale*(v_cond−v_null)`. Null не выбирает случайный ID.

In [ ]:
@torch.no_grad()
def teacher_microsteps(model, anchor, t, width, conditions, microsteps=2):
    state = anchor.detach()
    h = width / microsteps
    for k in range(microsteps):
        velocity = model(state, (float(t) - k * h) * TRAIN_TIMESTEPS,
                         conditions=conditions).sample
        state = state - h * velocity
    return state.detach()


def transition_features(anchor, endpoint, representation="endpoint", gamma=1.):
    dimension = endpoint[0].numel()
    a, y = anchor.flatten(1), endpoint.flatten(1)
    if representation == "endpoint":
        return y / math.sqrt(dimension)
    if representation == "displacement":
        return (y - a) / math.sqrt(dimension)
    if representation == "joint":
        return torch.cat((math.sqrt(gamma) * a, y - a), dim=1) / math.sqrt(dimension)
    raise ValueError("unknown representation")


@torch.no_grad()
def sample_conditioned(model, initial_noise, conditions, steps=32, guidance_scale=1.):
    if not isinstance(steps, int) or steps < 1:
        raise ValueError("steps must be a positive integer")
    if not math.isfinite(guidance_scale) or guidance_scale < 0:
        raise ValueError("guidance_scale must be finite and nonnegative")
    if initial_noise.ndim != 2 or initial_noise.shape[1] != 2 or len(initial_noise) < 1:
        raise ValueError("initial_noise must be [B,2]")
    labels = None
    if conditions is not None:
        labels = torch.as_tensor(conditions, device=model.device)
        if labels.ndim == 0:
            labels = labels.expand(len(initial_noise))
        if labels.dtype not in (torch.int32, torch.int64) or labels.shape != (len(initial_noise),):
            raise ValueError("conditions must be an integer target ID or [B] IDs")
        if (labels < 0).any() or (labels >= N_CLASSES).any():
            raise ValueError("target ID outside [0,N_CLASSES)")
    was_training = model.training
    model.eval()
    try:
        state = initial_noise.detach().to(device=model.device, dtype=model.dtype).clone()
        history = [state.cpu().clone()]
        for step in range(steps):
            t = 1. - step / steps
            conditional = model(state, t * TRAIN_TIMESTEPS, conditions=labels).sample
            if labels is None or guidance_scale == 1:
                velocity = conditional                 # direct null/conditional, CFG OFF
            else:
                unconditional = model(state, t * TRAIN_TIMESTEPS, conditions=None).sample
                velocity = unconditional + guidance_scale * (conditional - unconditional)
            state = state - velocity / steps
            if not torch.isfinite(state).all():
                raise FloatingPointError("Nonfinite conditional Euler trajectory")
            history.append(state.cpu().clone())
    finally:
        model.train(was_training)
    return {"samples": state.cpu(), "history": torch.stack(history),
            "times": torch.linspace(1, 0, steps + 1),
            "initial_noise_sha256": tensor_hash(initial_noise), "steps": steps,
            "nfe": steps * (1 if labels is None or guidance_scale == 1 else 2),
            "target": None if labels is None else labels.detach().cpu(),
            "guidance_scale": guidance_scale}


## Train-only calibration: отдельная таблица для каждого condition × interval

Для label `c` clean anchors берутся **только из train класса c**. Для null — из смеси.
Один support использует один label, одно время и одну frozen teacher. Epsilon рассчитывается
до students, по selected representation, и одинакова для tube/no-tube. Это не безусловная
epsilon, переиспользованная после разбиения на классы. Смена representation/parent требует
повторной калибровки и нового output directory.

In [ ]:
calibration_streams = RNGStreams(CALIBRATION_SEED)
calibration = {}
for condition_id in [None, *range(N_CLASSES)]:
    key = "null" if condition_id is None else str(condition_id)
    records = []
    pool = torch.arange(len(train)) if condition_id is None else class_indices[condition_id]
    for j in range(STUDENT_STEPS):
        selected = torch.randint(len(pool), (CALIBRATION_SAMPLES,), generator=calibration_streams["data"])
        indices = pool[selected]
        clean = train.samples[indices].to(device)
        noise = torch.randn(clean.shape, generator=calibration_streams["noise"]).to(device)
        labels = None if condition_id is None else torch.full((len(clean),), condition_id,
                                                            dtype=torch.long, device=device)
        t = 1 - j * H
        anchor = (1 - t) * clean + t * noise
        endpoint = teacher_microsteps(teacher, anchor, t, H, labels, MICROSTEPS)
        phi = transition_features(anchor, endpoint, REPRESENTATION, GAMMA)
        scale = calibrate_interval_features(phi.detach().cpu().double(), alpha=EPSILON_ALPHA)
        records.append({"interval": j, "t": t, "H": H, "condition": condition_id,
                        "anchor_sha256": tensor_hash(anchor), **scale})
    calibration[key] = records
calibration_metadata = {"teacher_sha256": teacher_hash, "train_sha256": tensor_hash(train.samples),
    "labels_sha256": tensor_hash(train.conditions), "representation": REPRESENTATION,
    "gamma": GAMMA, "alpha": EPSILON_ALPHA, "support_size": CALIBRATION_SAMPLES,
    "seed": CALIBRATION_SEED, "condition_interval_table": calibration}
calibration_hash = hashlib.sha256(json.dumps(calibration_metadata, sort_keys=True).encode()).hexdigest()
atomic_json(OUTPUT_DIR / "calibration.json", calibration_metadata | {"sha256": calibration_hash})
print("Frozen calibration:", len(calibration), "conditions including null ×", STUDENT_STEPS, "intervals")


## Видимый conditional student loop: M0 / M1 / E0 / E1

Сначала выбираются класс и null-флаг для **всего batch**, затем соответствующие clean samples.
На null-update clean samples снова смешанные. У четырёх методов одинаковые condition,
data, interval, noise и tube RNG; даже `rho=0` потребляет тот же `eta`. Все начинают с
точной копии teacher. Target после tube пересчитывается из perturbed input с тем же condition.
Смена условной задачи — exploratory-исследование; эти результаты не заменяют unconditional
таблицы репликации. `STUDENT_NULL_PROBABILITY=0` исключает null из distillation objective;
в этом случае student null-семплирование не следует считать проверенным обученным режимом.

In [ ]:
dist_label = "E" if (DISTRIBUTION_FAMILY, REPRESENTATION) == ("sinkhorn", "endpoint") else (
    DISTRIBUTION_FAMILY + "_" + REPRESENTATION + "_")
method_specs = [("M0", None, 0.), ("M1", None, TUBE_RHO),
                (dist_label + "0", DISTRIBUTION_FAMILY, 0.),
                (dist_label + "1", DISTRIBUTION_FAMILY, TUBE_RHO)]
trained, training_logs, pairing_traces = {}, {}, {}
for method_name, family, rho in method_specs:
    student = copy.deepcopy(teacher).train().requires_grad_(True)
    ema = copy.deepcopy(teacher).eval().requires_grad_(False)
    assert model_hash(student) == model_hash(ema) == teacher_hash
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE,
                                  betas=(.9, .999), weight_decay=WEIGHT_DECAY)
    streams = RNGStreams(SEED)
    condition_generator = torch.Generator().manual_seed(SEED + 9000)
    null_generator = torch.Generator().manual_seed(SEED + 9001)
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = method_dir / "training.pt"
    contract = {"scope": "exploratory_conditional_ray_and_null_distillation", "teacher_sha256": teacher_hash,
        "train_sha256": tensor_hash(train.samples), "labels_sha256": tensor_hash(train.conditions),
        "seed": SEED, "updates": UPDATES, "batch_size": BATCH_SIZE, "n_classes": N_CLASSES,
        "teacher_steps": TEACHER_STEPS, "student_steps": STUDENT_STEPS,
        "family": family, "representation": REPRESENTATION, "rho": rho,
        "null_probability": STUDENT_NULL_PROBABILITY, "lambda_w": LAMBDA_W, "gamma": GAMMA,
        "lr": LEARNING_RATE, "warmup": WARMUP, "weight_decay": WEIGHT_DECAY,
        "ema_decay": EMA_DECAY, "grad_clip": GRAD_CLIP,
        "swd_projections": SWD_PROJECTIONS, "calibration_sha256": calibration_hash}
    history, pairing_trace, completed = [], [], 0
    if checkpoint_path.exists():
        if not RESUME:
            raise FileExistsError(f"{checkpoint_path} exists: choose new OUTPUT_DIR or RESUME=True")
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if saved["contract"] != contract:
            raise ValueError("Conditional resume contract differs")
        student.load_state_dict(saved["student"])
        ema.load_state_dict(saved["ema"])
        optimizer.load_state_dict(saved["optimizer"])
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(device)
        streams.load_state_dict(saved["streams"])
        condition_generator.set_state(saved["condition_rng"])
        null_generator.set_state(saved["null_rng"])
        torch.set_rng_state(saved["torch_rng"])
        if device.type == "cuda" and saved["cuda_rng"]:
            torch.cuda.set_rng_state_all(saved["cuda_rng"])
        if device.type == "mps" and saved.get("mps_rng") is not None:
            torch.mps.set_rng_state(saved["mps_rng"])
        completed, history, pairing_trace = saved["updates_completed"], saved["history"], saved["pairing_trace"]

    def commit_checkpoint(status):
        save_checkpoint(checkpoint_path, {"contract": contract,
            "student": student.state_dict(), "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(), "streams": streams.state_dict(),
            "condition_rng": condition_generator.get_state(), "null_rng": null_generator.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
            "mps_rng": torch.mps.get_rng_state() if device.type == "mps" else None,
            "updates_completed": completed, "history": history,
            "pairing_trace": pairing_trace, "status": status})

    commit_checkpoint("running")
    try:
        for update in tqdm(range(completed, UPDATES), desc=method_name, disable=not PROGRESS):
            condition_id = int(torch.randint(N_CLASSES, (), generator=condition_generator))
            is_null = bool(torch.rand((), generator=null_generator) < STUDENT_NULL_PROBABILITY)
            pool = torch.arange(len(train)) if is_null else class_indices[condition_id]
            selected = torch.randint(len(pool), (BATCH_SIZE,), generator=streams["data"])
            indices = pool[selected]
            clean = train.samples[indices].to(device)
            labels = None if is_null else torch.full((BATCH_SIZE,), condition_id,
                                                     dtype=torch.long, device=device)
            j = int(torch.randint(STUDENT_STEPS, (), generator=streams["interval"]))
            t = 1 - j * H
            noise = torch.randn(clean.shape, generator=streams["noise"]).to(device)
            eta = torch.randn(clean.shape, generator=streams["tube"]).to(device)
            anchor = (1 - t) * clean + t * noise
            a_used = anchor + rho * H * eta
            teacher_endpoint = teacher_microsteps(teacher, a_used, t, H, labels, MICROSTEPS)
            student_endpoint = a_used - H * student(a_used, t * TRAIN_TIMESTEPS, conditions=labels).sample
            pair = (student_endpoint - teacher_endpoint.detach()).square().mean()
            dist = pair.new_zeros(())
            if family is not None and LAMBDA_W != 0:
                phi_student = transition_features(a_used, student_endpoint, REPRESENTATION, GAMMA)
                phi_teacher = transition_features(a_used, teacher_endpoint, REPRESENTATION, GAMMA).detach()
                calibration_key = "null" if is_null else str(condition_id)
                epsilon = calibration[calibration_key][j]["epsilon"]
                if family == "sinkhorn":
                    dist = production_sinkhorn(phi_student.float(), phi_teacher.float(), epsilon)
                elif family == "swd":
                    dist = sliced_wasserstein_loss(phi_student.float(), phi_teacher.float(),
                        num_directions=SWD_PROJECTIONS, generator=streams["swd"])
                else:
                    raise ValueError("unknown distribution family")
            loss = pair + LAMBDA_W * dist
            if not torch.isfinite(loss):
                raise FloatingPointError("Conditional student loss is nonfinite")
            lr = learning_rate(update, UPDATES, WARMUP, LEARNING_RATE)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP,
                                                       error_if_nonfinite=True)
            optimizer.step()
            with torch.no_grad():
                for average, parameter in zip(ema.parameters(), student.parameters()):
                    average.lerp_(parameter.detach(), 1 - EMA_DECAY)
                for average, buffer in zip(ema.buffers(), student.buffers()):
                    average.copy_(buffer)
            completed = update + 1
            pairing_trace.append(hashlib.sha256((f"{condition_id}:{is_null}:{j}:" + tensor_hash(indices)
                + tensor_hash(noise) + tensor_hash(eta)).encode()).hexdigest())
            history.append({"update": completed, "condition": None if is_null else condition_id,
                "interval": j, "pair_loss": float(pair.detach()), "dist_loss": float(dist.detach()),
                "total_loss": float(loss.detach()), "grad_norm": float(grad_norm), "lr": lr})
            if completed == 1 or completed % CHECKPOINT_EVERY == 0:
                commit_checkpoint("running")
    except BaseException as error:
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        saved["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
        save_checkpoint(checkpoint_path, saved)
        atomic_json(method_dir / "manifest.json", contract | {"status": saved["status"],
            "updates_completed": saved["updates_completed"], "error": str(error)})
        raise
    commit_checkpoint("completed")
    ema.save_pretrained(method_dir / "ema_model", safe_serialization=True)
    atomic_json(method_dir / "manifest.json", contract | {"status": "completed",
        "updates_completed": completed, "ema_sha256": model_hash(ema),
        "whole_support_single_condition": True, "null_updates": sum(r["condition"] is None for r in history)})
    atomic_json(method_dir / "history.json", history)
    assert model_hash(teacher) == teacher_hash and all(p.grad is None for p in teacher.parameters())
    trained[method_name], training_logs[method_name], pairing_traces[method_name] = ema, history, pairing_trace
assert all(trace == pairing_traces["M0"] for trace in pairing_traces.values())
print("All four methods: identical condition/null/data/t/noise/tube traces; frozen teacher unchanged.")


## Один noise bank → 16 разных target IDs и настоящий null

Каждый label получает **точную копию того же noise bank**. Таблица routing использует
полный requested-label phase error, confusion matrix и отдельную проверку запросов,
отличавшихся от исходного noise-arm. Wrong target не становится правильным после
переназначения labels. Null показывается отдельно; это смесь без target ID.

In [ ]:
initial_noise = torch.randn(SAMPLES_PER_TARGET, 2,
    generator=torch.Generator().manual_seed(EVAL_SEED))
models_to_compare = {"Teacher N32": (teacher, TEACHER_STEPS)}
models_to_compare.update({f"{name} N{STUDENT_STEPS}": (model, STUDENT_STEPS)
                          for name, model in trained.items()})
generations, routing_reports = {}, {}
for name, (model, steps) in models_to_compare.items():
    results = {}
    for target_id in [*range(N_CLASSES), None]:
        labels = None if target_id is None else torch.full((len(initial_noise),), target_id, dtype=torch.long)
        results[target_id] = sample_conditioned(model, initial_noise, labels, steps=steps, guidance_scale=1.)
    assert len({result["initial_noise_sha256"] for result in results.values()}) == 1
    generations[name] = results
    samples = torch.cat([results[label]["samples"] for label in range(N_CLASSES)])
    targets = torch.arange(N_CLASSES).repeat_interleave(len(initial_noise))
    repeated_noise = initial_noise.repeat(N_CLASSES, 1)
    routing_reports[name] = conditional_routing_metrics(samples, targets, repeated_noise, n_classes=N_CLASSES)
    print(name, "requested-ray accuracy:", routing_reports[name]["accuracy_all"],
          "outer accuracy:", routing_reports[name]["accuracy_outer"])
atomic_json(OUTPUT_DIR / "conditional_routing_validation.json", {
    "scope": "exploratory_conditional_same_noise_all_targets_and_null", "parent_sha256": teacher_hash,
    "noise_sha256": tensor_hash(initial_noise), "samples_per_target": len(initial_noise),
    "student_seed": SEED, "null_probability": STUDENT_NULL_PROBABILITY, "models": routing_reports})


In [ ]:
for name in (next(iter(generations)), list(generations)[-1]):
    histories = {label: result["history"] for label, result in generations[name].items()}
    fig, _ = plot_condition_routing(histories, validation.samples, validation.conditions,
                                   n_rays=N_CLASSES, noise_index=GIF_NOISE_INDEX)
    fig.suptitle(name + " · SAME initial noise, different requested target IDs")
    save_figure(fig, OUTPUT_DIR / (name.replace(" ", "_") + "_condition_routing"))
    plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name, history in training_logs.items():
    axes[0].plot([r["update"] for r in history], [r["pair_loss"] for r in history], label=name)
    axes[1].plot([r["update"] for r in history], [r["dist_loss"] for r in history], label=name)
axes[0].set(yscale="log", title="Conditional paired endpoint MSE", xlabel="Updates")
axes[1].set_yscale("symlog", linthresh=1e-7)
axes[1].set(title="Within-condition distribution penalty", xlabel="Updates")
for ax in axes:
    ax.legend()
save_figure(fig, OUTPUT_DIR / "conditional_losses")
plt.show()

gif_paths = []
if SAVE_GIFS:
    for name in (next(iter(generations)), list(generations)[-1]):
        histories = {label: result["history"] for label, result in generations[name].items()}
        path = OUTPUT_DIR / (name.replace(" ", "_") + "_target_routing.gif")
        save_condition_routing_gif(histories, validation.samples, validation.conditions, path,
                                  n_rays=N_CLASSES, noise_index=GIF_NOISE_INDEX)
        gif_paths.append(str(path))
        display(Image(filename=str(path)))
print("Target curves are highlighted; trajectory colors run red start → violet finish.")


## Интерактивный target, null, CFG и фактический Euler frame

Внутренний target dropdown включает 0…15 и **Unconditional**. Его Play/slider идут с шагом 1
по сохранённым Euler-узлам. Смена target внутри cache не запускает модель. CFG-checkbox ниже
изначально выключен; смена модели/CFG/steps применяется только кнопкой Generate и заново
строит все targets из одного неизменного initial noise. Значение CFG-slider сохраняется при
выключении. Никакого автоматического переноса частицы в нужный луч после generation нет.

In [ ]:
if SHOW_WIDGETS:
    model_selector = widgets.Dropdown(options=list(models_to_compare), description="Model")
    cfg_enabled = widgets.Checkbox(value=False, description="Use CFG")
    cfg_scale = widgets.FloatSlider(value=1.5, min=0., max=5., step=.1,
                                    description="CFG scale", disabled=True)
    inference_steps = widgets.IntSlider(value=TEACHER_STEPS, min=1, max=128, step=1,
                                        description="Euler steps", continuous_update=False)
    generate_button = widgets.Button(description="Generate all targets", button_style="primary")
    progress = widgets.IntProgress(value=0, min=0, max=N_CLASSES + 1, description="Targets")
    canvas = widgets.Output()

    def toggle_cfg(change=None):
        cfg_scale.disabled = not cfg_enabled.value

    def show_cached(name):
        histories = {label: result["history"] for label, result in generations[name].items()}
        with canvas:
            clear_output(wait=True)
            display(condition_routing_widget(histories, validation.samples, validation.conditions,
                                              n_rays=N_CLASSES, noise_index=GIF_NOISE_INDEX))

    def regenerate(change=None):
        generate_button.disabled = True
        try:
            model, _ = models_to_compare[model_selector.value]
            scale = cfg_scale.value if cfg_enabled.value else 1.0
            histories = {}
            progress.value = 0
            for index, target_id in enumerate([*range(N_CLASSES), None]):
                labels = None if target_id is None else torch.full((len(initial_noise),), target_id, dtype=torch.long)
                histories[target_id] = sample_conditioned(model, initial_noise, labels,
                    steps=inference_steps.value, guidance_scale=scale)["history"]
                progress.value = index + 1
            with canvas:
                clear_output(wait=True)
                print(model_selector.value, "CFG=", scale if cfg_enabled.value else "off",
                      "Euler steps=", inference_steps.value, "same initial noise for every target/null")
                display(condition_routing_widget(histories, validation.samples, validation.conditions,
                                                  n_rays=N_CLASSES, noise_index=GIF_NOISE_INDEX))
        finally:
            generate_button.disabled = False

    cfg_enabled.observe(toggle_cfg, names="value")
    generate_button.on_click(regenerate)
    display(widgets.VBox([widgets.HBox([model_selector, cfg_enabled, cfg_scale]),
                          widgets.HBox([inference_steps, generate_button, progress]), canvas]))
    show_cached(model_selector.value)


## Native reload и варианты

`DISTRIBUTION_FAMILY="swd"`, `REPRESENTATION="joint"` / `"displacement"`, `LAMBDA_W=0`
меняют видимые losses. Используйте новый output directory и всю четвёрку paired runs.
Чтобы проверить one-hot conditioner или другую архитектуру, задайте новый model profile,
`TRAIN_TEACHER=True`, новый каталог и обучите conditional teacher; существующие embedding
веса не являются one-hot pretrained model. Сложный continuous-conditioner требует явной
замены dataset conditions и соответствующего forward, как в notebook04/09.

Для 32→8 смените `STUDENT_STEPS=8`: microsteps, calibration и все student targets пересчитаются.
Не подменяйте unconditional replication этими conditional результатами. Здесь используем
свои train/validation seeds; официальные final/test артефакты не читаются и не изменяются.

In [ ]:
first_name = next(iter(trained))
restored = PointDenoiser.from_pretrained(OUTPUT_DIR / first_name / "ema_model",
    local_files_only=True, low_cpu_mem_usage=False).to(device).eval().requires_grad_(False)
test_labels = torch.zeros(len(initial_noise), dtype=torch.long)
restored_result = sample_conditioned(restored, initial_noise, test_labels, steps=STUDENT_STEPS)
assert torch.equal(restored_result["samples"], generations[f"{first_name} N{STUDENT_STEPS}"][0]["samples"])
assert model_hash(teacher) == teacher_hash
print("Native conditional EMA reload matches. Parent unchanged; condition enters every path.")
print("Outputs:", OUTPUT_DIR)
